# Notion API サンプル

Notion API（`notion-client`）の基本動作を確認するノートブック。

**前提条件**:
1. [https://www.notion.so/my-integrations](https://www.notion.so/my-integrations) で Internal Integration を作成済み
2. テスト用 Notion ページに Integration を接続済み（ページ右上「...」→「Connect to」）
3. `.env` に `NOTION_API_KEY` と `NOTION_TEST_PAGE_ID` を設定済み

In [1]:
from notion_client import Client
from dotenv import load_dotenv
import os

load_dotenv()

notion = Client(auth=os.environ["NOTION_API_KEY"])

me = notion.users.me()
print(f"Integration 名: {me['name']}")
print(f"Type         : {me['type']}")

Integration 名: c2n-test-itg
Type         : bot


## ページの取得

In [2]:
page_id = os.environ["NOTION_TEST_PAGE_ID"]

page = notion.pages.retrieve(page_id=page_id)

print(f"ページ ID   : {page['id']}")
print(f"作成日時    : {page['created_time']}")
print(f"最終更新日時: {page['last_edited_time']}")
print(f"URL         : {page['url']}")
print("\nプロパティ一覧:")
for key in page.get("properties", {}):
    print(f"  {key}")

ページ ID   : 36119e08-5037-803f-8226-f003174c51f1
作成日時    : 2026-05-15T14:30:00.000Z
最終更新日時: 2026-05-15T14:36:00.000Z
URL         : https://www.notion.so/Notion-API-Test-36119e085037803f8226f003174c51f1

プロパティ一覧:
  title


## ブロック（コンテンツ）の取得

In [3]:
blocks = notion.blocks.children.list(block_id=page_id)

print(f"ブロック数: {len(blocks['results'])}")
for block in blocks["results"]:
    btype = block["type"]
    rich_text = block.get(btype, {}).get("rich_text", [])
    text = "".join(t.get("plain_text", "") for t in rich_text) if rich_text else "(テキストなし)"
    print(f"  [{btype}] {text[:60]}")

ブロック数: 2
  [paragraph] (テキストなし)
  [child_page] (テキストなし)


## ページの作成

`NOTION_TEST_PAGE_ID` 配下に子ページを作成する。

In [4]:
new_page = notion.pages.create(
    parent={"page_id": page_id},
    properties={
        "title": {
            "title": [{"text": {"content": "notion_api_sample からの作成テスト"}}]
        }
    },
    children=[
        {
            "object": "block",
            "type": "paragraph",
            "paragraph": {
                "rich_text": [{"text": {"content": "このページは notion_api_sample.ipynb から API で作成されました。"}}]
            }
        }
    ]
)

print(f"作成されたページ ID: {new_page['id']}")
print(f"URL               : {new_page['url']}")

作成されたページ ID: 36119e08-5037-8157-9a88-f87326e48d1d
URL               : https://www.notion.so/notion_api_sample-36119e08503781579a88f87326e48d1d


## データベースのクエリ

`NOTION_TEST_DATABASE_ID` が設定されている場合のみ実行する。

In [5]:
database_id = os.environ.get("NOTION_TEST_DATABASE_ID")
if not database_id:
    print("NOTION_TEST_DATABASE_ID が未設定のためスキップします。")
else:
    results = notion.databases.query(database_id=database_id)
    print(f"取得件数: {len(results['results'])}")
    print(f"次ページあり: {results['has_more']}")
    for item in results["results"][:3]:
        print(f"  ID: {item['id']}, プロパティ: {list(item['properties'].keys())}")

NOTION_TEST_DATABASE_ID が未設定のためスキップします。
